# ArchiGen Facade V2: semantic ControlNet + UNet LoRA

Private academic experiment, separate from production. Preserve CMP 400/100/106 splits; add original IRFs human labels grouped by building URL and near-duplicate images. IRFs/ArchDaily is academic CC BY-NC-SA 4.0; attribution is retained. The four UI styles have no supervised labels.

Architecture: SD1.5 + 7-channel semantic ControlNet initialized from V1 step750 + rank8 UNet LoRA. Training uses aligned facade/storey crops, aspect-preserving letterbox, source balancing, FP16/checkpointing, opening-weighted Min-SNR loss. Select checkpoint by source-macro LPIPS on fixed 12 CMP + 4 IRF validation examples. Evaluate selected candidate and V1 on matched preprocessing; held-out tests untouched.

Run smoke first: two optimizer steps, validation, checkpoint, export and offloaded reload inference. This does not establish quality. Full mode: 2000 steps, full validation against parent, portable MLflow + local model bundle. One GPU is used; two T4 VRAM capacities do not add together.

In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
PINNED_COMMIT = ""
MODE = "smoke"
BRANCH = 'codex/facade-semantic-controlnet-v2'
REPO = Path('/kaggle/working/archigen-source')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/ayebkalil/archigen-3d.git',str(REPO)],check=True)
if not PINNED_COMMIT:
    raise ValueError('Stage this notebook with manage stage-kernel to pin a Git SHA')
subprocess.run(['git','fetch','--depth','1','origin',PINNED_COMMIT],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',PINNED_COMMIT],cwd=REPO,check=True)
print('Source commit:', PINNED_COMMIT, flush=True)
# Retain Kaggle native Torch/CUDA; only install model/tracking dependencies.
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'experiments/facade_semantic_v2/requirements.txt')],check=True)


In [ ]:
# Optional remote tracking / HF authentication belong in Kaggle Secrets.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ('HF_TOKEN','MLFLOW_TRACKING_URI','MLFLOW_TRACKING_TOKEN','MLFLOW_TRACKING_USERNAME','MLFLOW_TRACKING_PASSWORD'):
        try:
            value = secrets.get_secret(key)
            if value: os.environ[key] = value
        except Exception:
            pass
except ImportError:
    pass
from urllib.parse import urlsplit
if os.getenv('MLFLOW_TRACKING_URI') and urlsplit(os.environ['MLFLOW_TRACKING_URI']).hostname in {'localhost','127.0.0.1'}:
    raise ValueError('PC localhost is not reachable from Kaggle; remove URI for portable SQLite tracking')
subprocess.run([sys.executable,'-m','experiments.facade_semantic_v2.kaggle_run','--mode',MODE],cwd=REPO,check=True)


## Local use after evaluation
Download `archigen-facade-semantic-v2-model-and-mlflow.zip`, then use `python -m experiments.facade_semantic_v2.manage install --archive PATH.zip --destination models/saved/facade-semantic-v2 --tracking-uri sqlite:///mlflow.db` from this branch. Smoke bundles cannot be installed. The export contains ControlNet + LoRA and pins the frozen SD1.5 base. Use the separate diffusion environment and `SemanticFacadeRuntime`; CPU offload targets the RTX3050 but still needs actual local profiling. No production model is automatically replaced.

Full validation uses different geometry-preserving preprocessing from the published V1 benchmark. Only the matched V1/V2 comparison is meaningful. Opening-edge error is a proxy, not opening IoU. Neither visual plausibility nor a lower loss proves correct opening counts.